In [1]:
import cv2
import numpy as np
import pandas as pd
import glob
import os
from matplotlib import pyplot as plt
from tqdm import tqdm
from PIL import Image
from IPython.display import display, clear_output

In [3]:
# def select_corners(image_path, num_cells=1, window_size=(800,600)):
#     """Allows the user to click on multiple corners for selecting multiple solar cells."""
#     image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

#     if image is None:
#         print("Error loading image")
#         return None

#     selected_points = []

#     def click_event(event, x, y, flags, param):
#         if event == cv2.EVENT_LBUTTONDOWN:
#             selected_points.append((x, y))
#             cv2.circle(temp_image, (x, y), 5, (255, 0, 0), -1)
#             cv2.imshow("Select Corners", temp_image)
#             if len(selected_points) == num_cells:
#                 cv2.destroyAllWindows()

#     temp_image = image.copy()
#     cv2.namedWindow("Select Corners", cv2.WINDOW_NORMAL)
#     cv2.resizeWindow("Select Corners", *window_size)
#     cv2.imshow("Select Corners", temp_image)
#     cv2.setMouseCallback("Select Corners", click_event)
#     cv2.waitKey(0)

#     return selected_points if selected_points else None

In [2]:
def select_corners(image_path, num_cells=1, window_size=(800, 600)):
    """Allows the user to click on multiple corners for selecting multiple solar cells."""
    image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

    if image is None:
        print("Error loading image")
        return None

    # Enhance contrast using CLAHE (Contrast Limited Adaptive Histogram Equalization)
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image

    clahe = cv2.createCLAHE(clipLimit=6.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    selected_points = []

    def click_event(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            selected_points.append((x, y))
            cv2.circle(temp_image, (x, y), 5, (255, 0, 0), -1)
            cv2.imshow("Select Corners", temp_image)
            if len(selected_points) == num_cells:
                cv2.destroyAllWindows()

    temp_image = cv2.cvtColor(
        enhanced.copy(), cv2.COLOR_GRAY2BGR
    )  # Convert to BGR for color drawing
    cv2.namedWindow("Select Corners", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Select Corners", *window_size)
    cv2.imshow("Select Corners", temp_image)
    cv2.setMouseCallback("Select Corners", click_event)
    cv2.waitKey(0)

    return selected_points if selected_points else None

In [3]:
def extract_cells(image_path, corners, cell_size=(130, 85)):
    """Extracts multiple solar cells from an image given the selected corners."""
    image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

    if image is None:
        print(f"Error loading image: {image_path}")
        return None

    cells = []
    for corner in corners:
        x, y = corner
        w, h = cell_size
        cells.append(image[y : y + h, x : x + w])

    return cells

In [4]:
def compute_sector_stats(cell):
    """Computes mean and std for four sectors of the given cell."""
    h, w = cell.shape
    half_h, half_w = h // 2, w // 2

    sectors = {
        "top_left": cell[:half_h, :half_w],
        "top_right": cell[:half_h, half_w:],
        "bottom_left": cell[half_h:, :half_w],
        "bottom_right": cell[half_h:, half_w:],
        "whole": cell[:, :],
    }

    stats = {key: (np.mean(sector), np.std(sector)) for key, sector in sectors.items()}
    return stats

In [5]:
# def process_tiff_folder(folder_path, output_folder, reference_image=None, num_cells=1):
#     """Processes TIFF images in a folder, extracting multiple solar cells,
#     computing sector statistics, and plotting time series."""
#     os.makedirs(output_folder, exist_ok=True)

#     tiff_files = sorted(glob.glob(os.path.join(folder_path, f"*.tiff")))

#     if not tiff_files:
#         print(f"No TIFF files found.")
#         return

#     reference_image = reference_image if reference_image else tiff_files[0]
#     print(f"Using {reference_image} for corner selection.")
#     corners = select_corners(reference_image, num_cells)

#     if not corners or len(corners) != num_cells:
#         print("Not enough corners selected. Exiting.")
#         return

#     all_time_series = []

#     for cell_idx in range(num_cells):
#         time_series_data = {
#             "timestamp": [],
#             "mean_top_left": [],
#             "mean_top_right": [],
#             "mean_bottom_left": [],
#             "mean_bottom_right": []
#             }
#         std_series_data = {
#             "timestamp": [],
#             "std_top_left": [],
#             "std_top_right": [],
#             "std_bottom_left": [],
#             "std_bottom_right": []
#             }

#         for file in tqdm(tiff_files):
#             cells = extract_cells(file, corners)
#             if cells is None or len(cells) != num_cells:
#                 continue

#             cell = cells[cell_idx]
#             stats = compute_sector_stats(cell)
#             timestamp = os.path.basename(file).split("_")[1].split(".")[0]  # Extract timestamp from filename

#             time_series_data["timestamp"].append(int(timestamp)*3)
#             std_series_data["timestamp"].append(int(timestamp)*3)
#             for key in ["top_left", "top_right", "bottom_left", "bottom_right"]:
#                 time_series_data["mean_"+key].append(stats[key][0])  # Mean
#                 std_series_data["std_"+key].append(stats[key][1])   # Std deviation

#             filename = os.path.basename(file)
#             output_path = os.path.join(output_folder, f"cropped_cell{cell_idx+1}_{filename}")
#             cv2.imwrite(output_path, cell, [cv2.IMWRITE_TIFF_COMPRESSION, 1])
#             clear_output(wait=True)

#         display(f"Saved cropped cell {cell_idx+1}: {output_path}")

#         df_mean = pd.DataFrame(
#             time_series_data,
#             columns=[
#                 "timestamp",
#                 "mean_top_left",
#                 "mean_top_right",
#                 "mean_bottom_left",
#                 "mean_bottom_right"
#                 ])
#         df_mean = df_mean.sort_values(by="timestamp")
#         df_std = pd.DataFrame(
#             std_series_data,
#             columns=[
#                 "timestamp",
#                 "std_top_left",
#                 "std_top_right",
#                 "std_bottom_left",
#                 "std_bottom_right"
#                 ])
#         df_std = df_std.sort_values(by="timestamp")
#         df_std.drop(columns=["timestamp"], inplace=True)
#         df = pd.concat([df_mean, df_std], axis=1)
#         df.to_csv(os.path.join(output_folder, f"intensity_timeseries_cell{cell_idx+1}.csv"), index=False)
#         all_time_series.append((df, cell_idx+1))

#     for time_series_data, cell_idx in all_time_series:
#         plt.figure(figsize=(10, 5))
#         for key in ["top_left", "top_right", "bottom_left", "bottom_right"]:
#             _, _, bars = plt.errorbar(
#                 time_series_data["timestamp"],
#                 time_series_data[str("mean_"+key)],
#                 yerr=time_series_data[str("std_"+key)],
#                 fmt='-o', capsize=5, label=key
#                 )
#             [bar.set_alpha(0.5) for bar in bars]
#             # plt.plot(time_series_data["timestamp"], time_series_data["mean_"+key], marker='o', label=key)

#         plt.xlabel("Time [min]")
#         plt.ylabel("Mean Intensity")
#         plt.title(f"Solar Cell {cell_idx} Intensity Over Time by Sector")
#         plt.xticks(rotation=45)
#         plt.legend()
#         plt.grid()
#         plt.show()

In [7]:
def process_tiff_folder(
    folder_path, output_folder, reference_image=None, num_cells=1, max_measurements=None
):
    """Processes TIFF images in a folder, extracting multiple solar cells,
    computing sector statistics, and plotting time series."""
    os.makedirs(output_folder, exist_ok=True)

    # Get all TIFF files in the folder
    tiff_files = sorted(glob.glob(os.path.join(folder_path, "*.tiff")))

    if not tiff_files:
        print("No TIFF files found.")
        return

    # Extract the numeric part from the filenames to ensure proper sorting
    file_numbers = [
        int(os.path.basename(file).split("_")[3].split(".")[0]) for file in tiff_files
    ]

    # Sort files based on the numeric part of the filenames
    tiff_files_sorted = [x for _, x in sorted(zip(file_numbers, tiff_files))]

    # If max_measurements is specified, limit the number of files processed
    if max_measurements:
        tiff_files_sorted = tiff_files_sorted[:max_measurements]
        print(f"Limiting processing to the first {max_measurements} images.")

    reference_image = reference_image if reference_image else tiff_files_sorted[0]
    print(f"Using {reference_image} for corner selection.")
    corners = select_corners(reference_image, num_cells)

    if not corners or len(corners) != num_cells:
        print("Not enough corners selected. Exiting.")
        return

    all_time_series = []

    for cell_idx in tqdm(range(num_cells)):
        time_series_data = {
            "timestamp": [],
            "mean_top_left": [],
            "mean_top_right": [],
            "mean_bottom_left": [],
            "mean_bottom_right": [],
        }
        std_series_data = {
            "timestamp": [],
            "std_top_left": [],
            "std_top_right": [],
            "std_bottom_left": [],
            "std_bottom_right": [],
        }

        for file in tiff_files_sorted:
            cells = extract_cells(file, corners)
            if cells is None or len(cells) != num_cells:
                continue

            cell = cells[cell_idx]
            stats = compute_sector_stats(cell)
            timestamp = (
                os.path.basename(file).split("_")[3].split(".")[0]
            )  # Extract timestamp from filename

            time_series_data["timestamp"].append(int(timestamp) * 3)
            std_series_data["timestamp"].append(int(timestamp) * 3)
            for key in ["top_left", "top_right", "bottom_left", "bottom_right"]:
                time_series_data["mean_" + key].append(stats[key][0])  # Mean
                std_series_data["std_" + key].append(stats[key][1])  # Std deviation

            filename = os.path.basename(file)
            output_path = os.path.join(
                output_folder, f"cropped_cell{cell_idx+1}_{filename}"
            )
            cv2.imwrite(output_path, cell, [cv2.IMWRITE_TIFF_COMPRESSION, 1])
            clear_output(wait=True)

        display(f"Saved cropped cell {cell_idx+1}: {output_path}")

        df_mean = pd.DataFrame(
            time_series_data,
            columns=[
                "timestamp",
                "mean_top_left",
                "mean_top_right",
                "mean_bottom_left",
                "mean_bottom_right",
            ],
        )
        df_mean = df_mean.sort_values(by="timestamp")
        df_std = pd.DataFrame(
            std_series_data,
            columns=[
                "timestamp",
                "std_top_left",
                "std_top_right",
                "std_bottom_left",
                "std_bottom_right",
            ],
        )
        df_std = df_std.sort_values(by="timestamp")
        df_std.drop(columns=["timestamp"], inplace=True)
        df = pd.concat([df_mean, df_std], axis=1)
        # df.to_csv(os.path.join(f"substrate_D_cell{cell_idx+1}.csv"), index=False)
        all_time_series.append((df, cell_idx + 1))

    for time_series_data, cell_idx in all_time_series:
        fig = plt.figure(figsize=(10, 5))
        for key in ["top_left", "top_right", "bottom_left", "bottom_right"]:
            _, _, bars = plt.errorbar(
                time_series_data["timestamp"],
                time_series_data[str("mean_" + key)],
                yerr=time_series_data[str("std_" + key)],
                fmt="-o",
                capsize=5,
                label=key,
            )
            [bar.set_alpha(0.5) for bar in bars]
            # plt.plot(time_series_data["timestamp"], time_series_data["mean_"+key], marker='o', label=key)

        plt.xlabel("Time [min]")
        plt.ylabel("Mean Intensity")
        plt.title(f"Solar Cell {cell_idx} Intensity Over Time by Sector")
        plt.xticks(rotation=45)
        plt.legend()
        plt.grid()
        plt.show()

        fig.savefig(
            f"substrate_D_{cell_idx}",
            dpi=330,
            bbox_inches="tight",
        )

In [ ]:
# Example usage (adjust paths and exposure time as needed)
folder_path = "./data/PL_2603"  # Change this to your actual folder path
output_folder = "./cropped_cells"
process_tiff_folder(folder_path, output_folder, num_cells=16)

In [21]:
def plot_normalized_intensity(output_folder, prefix, num_cells=4):
    """Reads cropped cells, computes normalized intensity over time,
    and plots the evolution for all cells in one figure."""
    fig = plt.figure(figsize=(10, 5))
    cmap = plt.get_cmap("tab20")

    for cell_idx in tqdm(range(1, num_cells + 1)):
        cell_files = sorted(
            glob.glob(
                os.path.join(output_folder, f"cropped_cell{cell_idx}_{prefix}*.tiff")
            )
        )

        if not cell_files:
            print(f"No files found for Cell {cell_idx} with prefix {prefix}.")
            continue

        time_series = []
        intensities = []

        for file in cell_files:
            timestamp = (
                int(os.path.basename(file).split("_")[4].split(".")[0]) * 3
            )  # Extract timestamp and convert
            image = Image.open(file)
            mean_intensity = np.mean(np.array(image))

            time_series.append(timestamp)
            intensities.append(mean_intensity)

        time_series, intensities = zip(*sorted(zip(time_series, intensities)))
        normalized_intensities = np.array(intensities) / intensities[0]

        plt.plot(
            time_series,
            normalized_intensities,
            "-o",
            color=cmap(cell_idx - 1),
            label=f"Cell {cell_idx}",
        )

    plt.xlabel("Time [min]")
    plt.ylabel("Normalized Intensity")
    plt.title("Normalized Intensity Evolution Over Time for All Cells in Substrate D")
    plt.xticks(rotation=45)
    plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
    plt.grid()
    plt.show()

    fig.savefig(
        "substrate_D",
        dpi=330,
        bbox_inches="tight",
    )

In [ ]:
plot_normalized_intensity(output_folder, "mauri2_working", num_cells=4)